# Analisis Sentimen - Naive Bayes + TF-IDF
Dataset: Review produk dengan label `0` = Negatif, `1` = Netral, `2` = Positif

## 1. Import Library

In [ ]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from scipy.sparse import save_npz, load_npz
from google.colab import files

## 2. Upload & Load Dataset

In [ ]:
# Upload file Excel (data-bersih-teratur.xlsx)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# PERBAIKAN: file Excel harus pakai read_excel, bukan read_csv
df = pd.read_excel(filename)

# Hapus baris dengan label NaN
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("DATASET")
print(f"Shape: {df.shape}")
print(df.head())
print("\nDistribusi Label:")
print(df['label'].value_counts().sort_index())

## 3. TF-IDF Vectorization

In [ ]:
review = df['review_cleaned'].fillna('')
label = df['label']

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

# PERBAIKAN: X didefinisikan di sini supaya bisa dipakai di cell split
X = tfidf.fit_transform(review)

print(f"Shape matrix TF-IDF: {X.shape}")

# Simpan model dan matrix
joblib.dump(tfidf, 'model_tfidf.joblib')
save_npz('matrix_tfidf.npz', X)

print("Model dan matrix berhasil disimpan!")

## 4. Split Data Training & Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    label,
    test_size=0.2,
    random_state=42
)

print(f"Data Training : {X_train.shape[0]} baris")
print(f"Data Testing  : {X_test.shape[0]} baris")

## 5. Training Model Naive Bayes

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

print("Training model berhasil!")

## 6. Evaluasi Model

In [ ]:
prediksi = model.predict(X_test)

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, prediksi))

print("\nACCURACY")
print(accuracy_score(y_test, prediksi))

print("\nCLASSIFICATION REPORT")
# PERBAIKAN: target_names disesuaikan dengan 3 label yang ada
print(classification_report(y_test, prediksi, target_names=['Negatif', 'Netral', 'Positif']))

## 7. Prediksi Review Baru

In [ ]:
review_baru = [input("Masukkan review : ")]

review_vector = tfidf.transform(review_baru)
hasil = model.predict(review_vector)

# PERBAIKAN: label ada 3 (0, 1, 2) bukan hanya 2
label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

print(f"\nReview  : {review_baru[0]}")
print(f"Sentimen: {label_map[hasil[0]]}")

---
## (Opsional) Load Ulang Model Jika Session Baru

In [ ]:
# Jalankan cell ini HANYA jika session Colab sudah expired / restart
# Upload ketiga file: data-bersih-teratur.xlsx, model_tfidf.joblib, matrix_tfidf.npz
uploaded = files.upload()

df      = pd.read_excel('data-bersih-teratur.xlsx')
df      = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

tfidf        = joblib.load('model_tfidf.joblib')
X            = load_npz('matrix_tfidf.npz')    # load_npz, bukan joblib.load
label        = df['label']

print("Model:", tfidf)
print("Matrix shape:", X.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))